# Isolation Forest: Mathematical Framework

## Core Principle

Isolation Forest identifies anomalies by isolating observations through random partitioning. The fundamental insight is that **anomalies are easier to isolate** than normal points.

## Mathematical Definitions

### 1. Path Length $h(x)$
For a point $x$, the path length is the number of edges traversed from the root to a leaf node in an isolation tree.

### 2. Average Path Length $c(n)$
The normalization factor for a sample of size $n$:

$$c(n) = \begin{cases}
2H_{n-1} - \frac{2(n-1)}{n} & \text{if } n > 2 \\
1 & \text{if } n = 2 \\
0 & \text{otherwise}
\end{cases}$$

Where $H_{n-1}$ is the harmonic number:
$$H_{n-1} = \ln(n-1) + \gamma$$

$\gamma \approx 0.5772156649$ is Euler's constant.

### 3. Anomaly Score $s(x, n)$

$$s(x, n) = 2^{-\frac{\mathbb{E}[h(x)]}{c(n)}}$$

Where $\mathbb{E}[h(x)]$ is the average path length across all trees in the ensemble.

### 4. Score Interpretation

- $s \approx 1$: **Anomaly** (short paths)
- $s \approx 0.5$: **Normal point** (average paths)
- $s \approx 0$: **Normal point** (long paths)

### Algorithm Parameters

- **Sample size** $\psi$: Typically 256
- **Number of trees** $t$: Typically 100
- **Height limit** $l = \lceil \log_2(\psi) \rceil$

### Training Process

1. Subsample $\psi$ points from training data
2. Build isolation tree with height limit $l$
3. Repeat $t$ times to create ensemble
4. For each tree, recursively partition by random feature and split value

### Scoring Process

1. For each point, compute path length in each tree
2. Adjust by $c(\text{leaf\_size})$
3. Average across ensemble
4. Compute anomaly score using the formula above

## Implementation - Helper Function and Node Classes

In [7]:
import numpy as np
import pandas as pd

def c(size):
    """Average path length normalization factor c(n).
    
    For n > 2:  c(n) = 2 * H(n-1) - 2*(n-1)/n
    where H(i) = ln(i) + 0.5772156649 (Euler's constant)
    """
    if size > 2:
        return 2 * (np.log(size-1) + 0.5772156649) - 2*(size-1)/size
    if size == 2:
        return 1
    return 0

class LeafNode:
    """Terminal node containing isolated points."""
    def __init__(self, size, data):
        self.size = size
        self.data = data

class DecisionNode:
    """Internal node with splitting rule."""
    def __init__(self, left, right, splitAtt, splitVal):
        self.left = left
        self.right = right
        self.splitAtt = splitAtt
        self.splitVal = splitVal

## Isolation Tree Implementation

In [8]:
class IsolationTree:
    """Single isolation tree using random recursive partitioning."""
    
    def __init__(self, height, height_limit):
        self.height = height
        self.height_limit = height_limit
        self.root = None

    def fit(self, X: np.ndarray):
        """Build isolation tree until height limit or sample size ≤ 2."""
        if self.height >= self.height_limit or X.shape[0] <= 2:
            self.root = LeafNode(X.shape[0], X)
            return self.root
        
        # Randomly select feature and split value
        num_features = X.shape[1]
        splitAtt = np.random.randint(0, num_features)
        splitVal = np.random.uniform(min(X[:, splitAtt]), max(X[:, splitAtt]))
        
        # Partition data
        X_left = X[X[:, splitAtt] < splitVal]
        X_right = X[X[:, splitAtt] >= splitVal]
        
        # Recursively build subtrees
        left = IsolationTree(self.height + 1, self.height_limit)
        right = IsolationTree(self.height + 1, self.height_limit)
        left.fit(X_left)
        right.fit(X_right)
        
        self.root = DecisionNode(left.root, right.root, splitAtt, splitVal)
        return self.root

## Isolation Forest Ensemble

In [9]:
class IsolationTreeEnsemble:
    """Ensemble of isolation trees for anomaly detection."""
    
    def __init__(self, sample_size, n_trees=100):
        self.sample_size = sample_size
        self.n_trees = n_trees
        self.trees = []

    def fit(self, X: np.ndarray):
        """Build ensemble by sampling subsets of data."""
        if isinstance(X, pd.DataFrame):
            X = X.values
            
        n_rows = X.shape[0]
        height_limit = np.ceil(np.log2(self.sample_size))
        
        for _ in range(self.n_trees):
            # Subsample with replacement
            data_index = np.random.randint(0, n_rows, self.sample_size)
            X_sub = X[data_index]
            
            tree = IsolationTree(0, height_limit)
            tree.fit(X_sub)
            self.trees.append(tree)
        
        return self

    def path_length(self, X: np.ndarray) -> np.ndarray:
        """Compute average path length for each point across all trees."""
        if isinstance(X, pd.DataFrame):
            X = X.values
            
        avg_paths = []
        for row in X:
            lengths = []
            for tree in self.trees:
                node = tree.root
                length = 0
                
                # Traverse tree
                while isinstance(node, DecisionNode):
                    if row[node.splitAtt] < node.splitVal:
                        node = node.left
                    else:
                        node = node.right
                    length += 1
                
                # Add normalization factor
                total_length = length + c(node.size)
                lengths.append(total_length)
            
            avg_paths.append(np.mean(lengths))
        
        return np.array(avg_paths)

    def anomaly_score(self, X) -> np.ndarray:
        """Compute anomaly scores: s(x,n) = 2^(-E[h(x)]/c(n))"""
        if isinstance(X, pd.DataFrame):
            X = X.values
            
        avg_paths = self.path_length(X)
        scores = 2 ** (-avg_paths / c(self.sample_size))
        return scores

    def predict(self, X, threshold: float) -> np.ndarray:
        """Convert anomaly scores to binary predictions."""
        scores = self.anomaly_score(X)
        return np.array([1 if s >= threshold else 0 for s in scores])

# Implementation Summary

## Algorithm Steps

### Training Phase
1. **Subsample**: Randomly select ψ points from training data
2. **Build Tree**: Recursively partition until height limit or leaf size ≤ 2
   - Randomly choose a feature
   - Randomly choose split value between min and max of that feature
3. **Ensemble**: Repeat t times to build forest

### Scoring Phase  
1. **Path Length**: For each point, traverse all trees to compute path length
2. **Normalize**: Add c(leaf_size) to path length from each tree
3. **Average**: Calculate mean path length across ensemble
4. **Score**: Apply formula s = 2^(-E[h]/c(ψ))

## Key Properties

- **Time Complexity**: O(t·ψ·log ψ) for training
- **Space Complexity**: O(t·ψ) for storage
- **No distance calculations** - scales well to high dimensions
- **Unsupervised** - no need for labeled anomalies

## Scoring Threshold Interpretation

| Score Range | Interpretation |
|-------------|----------------|
| > 0.6 | Likely anomaly |
| 0.5 ± 0.1 | Normal point |
| < 0.4 | Very normal |

The threshold can be adjusted based on desired false positive rate.